# Smart MCQ Solver

## Submission By:

- **Name:** Varun Agnihotri
- **Roll Number:** `24f2004142`

In [ ]:
import os
import random
import sys
from pathlib import Path

import pandas as pd
import torch

# Model Selection: "local" (BiLSTM) or "finetune" (Gemma-4 Fine-tuned)
MODEL_MODE = "finetune"

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")

if IS_KAGGLE:
    print("Running in Kaggle environment.")
    TEST_CSV_PATH = Path(
        "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
    )
    SUBMISSION_OUT_PATH = Path("submission.csv")

    # Resolve Model Registry
    base_model_dir = Path(
        "/kaggle/input/models/pythonicvarun/smart-mcq-solver/pytorch/local"
    )
    if base_model_dir.exists():
        version_dirs = list(base_model_dir.glob("*"))
        LOCAL_MODEL_DIR = version_dirs[0] if version_dirs else base_model_dir
    else:
        LOCAL_MODEL_DIR = Path(
            "/kaggle/input/models/pythonicvarun/smart-mcq-solver/pytorch/local/2"
        )

    SRC_PATH = LOCAL_MODEL_DIR / "src"

    if MODEL_MODE == "finetune":
        MODEL_DIR = Path(
            "/kaggle/input/models/pythonicvarun/smart-mcq-solver/transformers/gemma4-lora-finetune/1"
        )
    else:
        MODEL_DIR = LOCAL_MODEL_DIR
else:
    print("Running in local environment.")
    if Path(os.getcwd()).name == "notebooks":
        os.chdir("..")

    print(f"Current working directory: {os.getcwd()}")

    TEST_CSV_PATH = Path("dataset/test.csv")
    SRC_PATH = Path("src")
    if MODEL_MODE == "finetune":
        MODEL_DIR = Path("outputs/gemma4-lora")
        SUBMISSION_OUT_PATH = MODEL_DIR / "submission_gemma_model.csv"
    else:
        MODEL_DIR = Path("outputs/local")
        SUBMISSION_OUT_PATH = MODEL_DIR / "submission_local_model.csv"

# add source path to python search path
if SRC_PATH.exists():
    if str(SRC_PATH.resolve()) not in sys.path:
        sys.path.append(str(SRC_PATH.resolve()))
    print(f"Added source path to sys.path: {SRC_PATH.resolve()}")
else:
    print(
        f"WARNING: Source path '{SRC_PATH}' not found. Make sure packages are importable."  # noqa: E501
    )

print(f"Test CSV path: {TEST_CSV_PATH}")
print(f"Model directory: {MODEL_DIR}")
print(f"Submission output path: {SUBMISSION_OUT_PATH}")

In [ ]:
if MODEL_MODE == "finetune":
    # Install dependencies for fine-tuned model inference (when MODEL_MODE = "finetune")
    !uv pip install unsloth unsloth-zoo triton transformers \
        peft bitsandbytes trl datasets

In [ ]:
from local import config

# override config directories
config.OUTPUT_DIR = MODEL_DIR
config.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device in use: {config.DEVICE}")

In [ ]:
import pickle
import re
from typing import Dict, List

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
from unsloth import FastLanguageModel


def clean(text: str) -> str:
    text = str(text).lower()  # type: ignore
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def extract_core_question(prompt: str) -> str:
    """Extract core question by removing template prefixes and suffixes."""
    if ":" in prompt:
        q = prompt.split(":", 1)[1].strip()
    else:
        q = prompt.strip()

    if "?" in q:
        q = q.rsplit("?", 1)[0].strip() + "?"
    return clean(q)


class TFIDFRetriever:
    """
    TF-IDF retriever over the training corpus that provides few-shot
    context for each question.
    """

    def __init__(self, df: pd.DataFrame | pd.Series, top_k: int = 3) -> None:
        self.top_k = top_k
        self.corpus = df.reset_index(drop=True).copy()
        self.corpus["_core"] = self.corpus["prompt"].apply(extract_core_question)

        self.vectorizer = TfidfVectorizer(
            max_features=50_000,
            ngram_range=(1, 2),
            sublinear_tf=True,
            strip_accents="unicode",
            token_pattern=r"(?u)\b\w\w+\b",
        )
        self.corpus_matrix = self.vectorizer.fit_transform(self.corpus["prompt"])
        print(
            f"TF-IDF index: {self.corpus_matrix.shape[0]} documents, "
            f"{self.corpus_matrix.shape[1]} features"
        )

    def retrieve(self, query: str) -> str:
        """
        Returns a formatted context string for one query prompt.

        Returned format (3 retrieved pairs, separated by blank lines):
          Q: <similar question>
          Answer: <label>) <full answer text>

          Q: ...
          Answer: ...
        """
        core_q = extract_core_question(query)
        qv = self.vectorizer.transform([core_q])
        sims = cosine_similarity(qv, self.corpus_matrix).flatten()

        order = sims.argsort()[::-1]
        ctx_parts = []
        for idx in order:
            row = self.corpus.iloc[idx]
            if row["_core"] == core_q:
                continue

            label = str(row["answer"]).strip().upper()
            answer_text = str(row[label]).strip()
            ctx_parts.append(f"Q: {row['prompt']}\nAnswer: {label}) {answer_text}")
            if len(ctx_parts) >= self.top_k:
                break

        return "\n\n".join(ctx_parts)

    def precompute_contexts(self, df: pd.DataFrame | pd.Series, desc: str) -> List[str]:
        return [
            self.retrieve(row["prompt"])
            for _, row in tqdm(df.iterrows(), total=len(df), desc=desc)
        ]

    def save(self, path: str | Path):
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path: str | Path):
        with open(path, "rb") as f:
            return pickle.load(f)


def build_user_message(row: Dict, context: str = "") -> str:
    """
    Builds the user-turn message.

    With context (self RAG):
      Context:
      Q: <similar question>
      Answer: A) <text>

      Q: ...
      Answer: ...

      Question: <current question>
      A) ...   B) ...   C) ...   D) ...   E) ...
      The correct answer is:

    Without context (zero-shot):
      Question: <current question>
      ...
    """
    context_block = f"Context:\n{context.strip()}\n\n" if context.strip() else ""
    return (
        f"{context_block}"
        f"Question: {row['prompt']}\n\n"
        f"A) {row['A']}\n"
        f"B) {row['B']}\n"
        f"C) {row['C']}\n"
        f"D) {row['D']}\n"
        f"E) {row['E']}\n\n"
        f"The correct answer is:"
    )


def get_choice_token_ids(tokenizer, choices: List[str]) -> Dict[str, int]:
    """
    Resolves each answer label to its single vocabulary token ID.
    """
    choice_ids = {}
    for ch in choices:
        plain = tokenizer.tokenizer.encode(ch, add_special_tokens=False)
        spaced = tokenizer.tokenizer.encode(" " + ch, add_special_tokens=False)
        if len(plain) == 1:
            choice_ids[ch] = plain[0]
        elif len(spaced) == 1:
            choice_ids[ch] = spaced[0]
        else:
            choice_ids[ch] = plain[0]
            print(f"Warning: '{ch}' is multi-token ({plain}). Using first token.")

    decoded = {k: (v, tokenizer.tokenizer.decode([v])) for k, v in choice_ids.items()}
    print(f"Choice token IDs: {decoded}")
    return choice_ids


@torch.inference_mode()
def score_and_rank(
    model,
    tokenizer,
    row: Dict,
    choice_token_ids: Dict[str, int],
    choices: List[str],
) -> List[str]:
    """
    Single forward pass - log-prob ranking of all 5 choices.
    """
    context = str(row.get("_context", ""))
    messages = [{"role": "user", "content": build_user_message(row, context)}]
    prefix_str = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # appends <start_of_turn>model\n
    )
    # print(prefix_str)
    input_ids = tokenizer(
        text=prefix_str,
        return_tensors="pt",
        add_special_tokens=False,
    ).input_ids.to(model.device)

    # Forward pass
    last_logits = model(input_ids).logits[0, -1, :]  # [vocab_size]
    log_probs = torch.log_softmax(last_logits, dim=-1)

    scores = {ch: log_probs[tid].item() for ch, tid in choice_token_ids.items()}
    ranked = sorted(scores, key=lambda k: scores[k], reverse=True)
    return ranked[:3]


def load_artifacts():
    if MODEL_MODE == "finetune":
        print(f"Loading finetuned model artifacts from: {MODEL_DIR}")
        retriever = TFIDFRetriever.load(os.path.join(MODEL_DIR, "retriever.pkl"))
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=str(MODEL_DIR),
            max_seq_length=512,
            dtype=None,
            load_in_4bit=True,
            device_map={"": 0},
        )
        FastLanguageModel.for_inference(model)
        return model, tokenizer, retriever
    else:
        from local.inference import load_artifacts as load_local_artifacts

        return load_local_artifacts()


def predict(df, model, retriever, vocab, is_train=False):
    if MODEL_MODE == "finetune":
        model.eval()
        choices = ["A", "B", "C", "D", "E"]
        choice_ids = get_choice_token_ids(vocab, choices)

        preds = []
        print("Pre-computing contexts for prediction...")
        contexts = [
            retriever.retrieve(row["prompt"])
            for _, row in tqdm(df.iterrows(), total=len(df), desc="Retrieving contexts")
        ]

        for idx, (_, row) in enumerate(
            tqdm(df.iterrows(), total=len(df), desc="Generating predictions")
        ):
            row_dict = row.to_dict()
            row_dict["_context"] = contexts[idx]
            top3 = score_and_rank(model, vocab, row_dict, choice_ids, choices)
            preds.append(" ".join(top3))

        sub = pd.DataFrame({"id": df.get("id", range(len(df))), "Prediction": preds})
        return sub
    else:
        from local.inference import predict as predict_local

        return predict_local(df, model, retriever, vocab, is_train=is_train)


# Check test file
if not TEST_CSV_PATH.exists():
    print("Required test dataset not found, exiting!")
    exit(1)

test_df = pd.read_csv(TEST_CSV_PATH)
print(f"Loaded test dataset with {len(test_df)} records.")

In [ ]:
# Load model, vocab, and retriever
model, vocab, retriever = load_artifacts()

model

In [ ]:
# Run prediction pipeline
sub_df = predict(test_df, model, retriever, vocab, is_train=False)

# Ensure directory exists for output if running locally
SUBMISSION_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

sub_df.to_csv(SUBMISSION_OUT_PATH, index=False)
print(f"Saved submission to: {SUBMISSION_OUT_PATH.resolve()}")

In [ ]:
sub_df.head(10)

In [ ]:
# Sample Visualization
random.seed(42)

sample_indices = random.sample(range(len(test_df)), min(3, len(test_df)))

for idx in sample_indices:
    row = test_df.iloc[idx]
    pred = sub_df.iloc[idx]["Prediction"]

    print("-" * 120)
    print(f"[ID {row['id']}] PROMPT:")
    print(row["prompt"])
    print("-" * 60)
    for c in ["A", "B", "C", "D", "E"]:
        print(f"({c}) {row[c]}")
    print("-" * 60)

    if MODEL_MODE == "finetune":
        ctx = retriever.retrieve(row["prompt"])
    else:
        ctx = retriever.retrieve(row["prompt"], top_k=1)

    print(f"Retrieved Context (Top-1): {ctx[:300]}...")
    print("-" * 60)
    print(f"Prediction (MAP@3 Order): {pred}")
    print("-" * 120)
    print()